# Fog-density object detection: full training and comparison

Train **YOLO11n, YOLO11m and DETR** on all prepared training images, with and without an estimated fog-density input. Detector weights and the matched RGB conditioning adapter are jointly optimized. The default experiment runs 30 epochs and three seeds per model/arm; it is a substantial GPU job.

**Run All now runs training when preflight is ready.** Missing data produces a specific blocking checklist. Training, preparation and prediction exports resume after interruption. Set `ACTION="analyze"` to reopen existing results without retraining.

This notebook has no completed full-study results yet. Verification outputs and small test fixtures are not scientific findings.


## 1. Configure the experiment

Use the project's Python kernel. All paths are rooted in this workspace. The GPU is selected automatically. The complete nuScenes trainval image release is required; preparation and preflight stop until every labeled camera keyframe is available. There is no frame cap.

The default two arms are `rgb_control` (constant-zero scalar) and `predicted_density` (RGB-estimated beta). Both train the same adapter architecture and detector with identical initial weights, seeds, data, epochs and optimizer settings. The optional `oracle_density` arm requires beta labels for **every** matched image; native nuScenes does not provide them.


In [1]:
from pathlib import Path
import os, sys, json
import pandas as pd
from IPython.display import display, Image

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT/'notebooks/_support/fog_training.py').is_file(), 'Launch Jupyter from this project'
os.environ.setdefault('YOLO_CONFIG_DIR', str(ROOT/'.ultralytics'))
os.environ.setdefault('HF_HOME', str(ROOT/'.cache/huggingface'))
os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')
sys.path.insert(0, str(ROOT/'notebooks/_support'))
from fog_training import default_config, preflight, run_training
from fog_research import read_manifest
from fog_prepare import prepare_nuscenes
from fog_data import cityscapes_manifest, resolve_foggy_images
from fog_analytics import export_analytics

ACTION = 'train'  # 'train', 'preflight', or 'analyze'
CONFIG = default_config(ROOT)
CONFIG['epochs'] = 30
CONFIG['seeds'] = [42, 43, 44]
CONFIG['size'] = 640
CONFIG['accumulate'] = 4  # one image per GPU microstep, four accumulated gradients
CONFIG['arms'] = ['rgb_control', 'predicted_density']
CONFIG['density_checkpoint'] = None  # or an independently trained fog-density-v1 checkpoint
CONFIG['output'] = str(ROOT/'reports/fog/full-training')  # same path/config resumes; new experiment needs new path
CITY_ROOT = ROOT/'data/fog/cityscapes'
DOWNLOAD_FOG_IMAGES = True  # Automatically download ~29.1 GB when Foggy Cityscapes is absent
NUSCENES_ROOT = ROOT/'data/depth/datasets/nuscenes'
assert ACTION in ('train','preflight','analyze')
display(pd.DataFrame([{'setting':k, 'value':str(v)} for k,v in CONFIG.items()]))

,setting,value
0,output,C:\Users\tsao1\OneDrive\Documents\workspace\Ai...
1,manifest,C:\Users\tsao1\OneDrive\Documents\workspace\Ai...
2,city_manifest,C:\Users\tsao1\OneDrive\Documents\workspace\Ai...
3,density_checkpoint,None
4,checkpoints,{'YOLO11n': 'C:\\Users\\tsao1\\OneDrive\\Docum...
5,arms,"['rgb_control', 'predicted_density']"
6,seeds,"[42, 43, 44]"
7,device,cuda:0
8,size,640
9,epochs,30


## 2. Prepare or reuse all data

nuScenes preparation streams metadata into a local index and checkpoints each scene. It accounts for all six cameras and all 34,149 labeled trainval samples, not unannotated sweeps. Missing files are listed in `nuscenes_all.coverage.json`; re-run preparation after extracting the remaining blobs. Preparation refuses to produce a study manifest while any expected keyframe is missing.

Official training scenes are deterministically divided into detector training (~90%) and tuning (~10%); official validation scenes serve as final test. All cameras from one scene remain together. Test scenes are never used for optimization or checkpoint selection.

Your supplied `Untitled2.ipynb` downloaded `minhducnguyen9705/foggy-cityscapes` version 1 to a Linux Kaggle cache and verified 15,000 images. This notebook now reuses an existing copy or automatically downloads the same source when it is absent. Set `DOWNLOAD_FOG_IMAGES=False` only when you intentionally want a no-download preflight. Existing project, Downloads and home Kaggle caches are detected automatically.

Images alone suffice: beta comes from filenames and nuScenes supplies detection boxes. Cityscapes `gtFine` is **not required** for this workflow. Images without box labels are explicitly marked density-only and never treated as empty detection ground truth. Foggy Cityscapes train sequences are split into density training (80%) and tuning (20%); original val is held out for density error. Optional detection benchmarking on Foggy Cityscapes requires a separate manifest made with `with_detection_labels=True` and gtFine annotations.


In [2]:
if ACTION != 'analyze':
    if not Path(CONFIG['manifest']).exists():
        prepare_nuscenes(NUSCENES_ROOT, CONFIG['manifest'])
    if not Path(CONFIG['city_manifest']).exists():
        city_source = resolve_foggy_images(ROOT, CITY_ROOT, download=DOWNLOAD_FOG_IMAGES)
        if city_source is not None:
            cityscapes_manifest(city_source, CONFIG['city_manifest'])
        else:
            print('Foggy Cityscapes source not found. Set CITY_ROOT to a directory containing leftImg8bit_foggy/train and val, or set DOWNLOAD_FOG_IMAGES=True. The density comparison will remain blocked until cityscapes.json is created.')

    rows = []
    for name in ('manifest','city_manifest'):
        path = Path(CONFIG[name])
        if path.exists():
            print('Validating', path.name)
            rows.extend(read_manifest(path))
        else:
            print('Not present:', path)
    if rows:
        frame = pd.DataFrame(rows)
        display(frame.groupby(['dataset','split']).agg(images=('id','size'), scenes=('group','nunique'), known_beta=('beta','count')))
        display(frame[frame.dataset == 'nuscenes'].groupby(['split','camera']).size().rename('images').reset_index())
    coverage_path = Path(CONFIG['manifest']).with_suffix('.coverage.json')
    if coverage_path.exists():
        coverage = json.loads(coverage_path.read_text())
        display(pd.DataFrame([{'expected_images':coverage['expected_images'], 'available_images':coverage['available_images'],
                              'missing_images':len(coverage['missing_images']), 'complete':coverage['complete']}]))

Validating nuscenes_all.json
Validating cityscapes.json


images  scenes  known_beta
dataset          split                                     
foggy_cityscapes density_train     6816    1527        6816
                 test              1500     235        1500
                 val               2109     358        2109
nuscenes         detector_train  150948     626           0
                 test             36114     150           0
                 val              17832      74           0

,split,camera,images
0,detector_train,CAM_BACK,25158
1,detector_train,CAM_BACK_LEFT,25158
2,detector_train,CAM_BACK_RIGHT,25158
3,detector_train,CAM_FRONT,25158
4,detector_train,CAM_FRONT_LEFT,25158
5,detector_train,CAM_FRONT_RIGHT,25158
6,test,CAM_BACK,6019
7,test,CAM_BACK_LEFT,6019
8,test,CAM_BACK_RIGHT,6019
9,test,CAM_FRONT,6019


,expected_images,available_images,missing_images,complete
0,204894,204894,0,True


## 3. Preflight

The full comparison requires an independently trained density estimator or Foggy Cityscapes images with beta in their filenames. If only the RGB baseline is desired, explicitly set `CONFIG['arms']=['rgb_control']` and choose a separate output folder. No random estimator or fabricated density is used as a fallback.

An external checkpoint must use this pipeline's density format and declare its training groups/units. Directly loading an unrelated model's weights is unsupported until an adapter and calibration are supplied.


In [3]:
checks = None
if ACTION != 'analyze':
    checks = preflight(CONFIG, rows)
    display(checks)
    if not checks.ready.all():
        print('BLOCKED: resolve the following inputs/settings before full training:')
        display(checks.loc[~checks.ready, ['check','detail']])

,check,ready,detail
0,training images,True,150948 labeled images
1,tuning images,True,17832 scene-held-out images for checkpoint sel...
2,test images,True,36114 held-out images for final metrics
3,arms,True,"rgb_control, predicted_density"
4,scene/ID isolation,True,"No scene crosses train, tuning, density-traini..."
5,size,True,640
6,epochs,True,30
7,density_epochs,True,15
8,accumulate,True,4
9,save_every,True,200


## 4. Train, resume and evaluate

Every model/arm starts from its original pretrained weights; training one arm cannot contaminate the next. All trainable detector parameters and the adapter are optimized. BatchNorm running statistics are kept fixed for stable single-image microsteps; affine parameters remain trainable. AMP and gradient accumulation reduce GPU memory demand. Checkpoint selection uses tuning loss, never final-test accuracy.

The run saves optimizer/scaler/RNG state every configured number of optimizer steps and at each epoch. Interrupting rolls back at most to the last saved update. Rerunning with identical settings resumes; changes to code, inputs or settings require a new output folder. Completed model/arm runs are reused. File locks prevent two kernels from writing the same run.

Final evaluation uses batch-one FP32, three warm-ups, CUDA synchronization, and a saved prediction record per image. Latency includes JPEG decode, resize, GPU transfer, density estimation, adapter, detector and postprocessing. Downloads and training are excluded. Hardware, caching and other workloads affect latency; serial FPS is not video throughput.


In [ ]:
results = pd.DataFrame()
if ACTION == 'train':
    if checks.ready.all():
        results, status = run_training(rows, CONFIG)
        display(status)
    else:
        print('Training did not start because preflight failed. See the explicit blockers above.')
elif ACTION == 'analyze':
    results = export_analytics(CONFIG['output'])
else:
    print('Preflight-only mode selected. Set ACTION="train" when ready.')

Density seed=42 epoch=2:   0%|          | 0/2016 [00:00<?, ?it/s]

Density tuning:   0%|          | 0/2109 [00:00<?, ?it/s]

Density seed=42 epoch=3:   0%|          | 0/6816 [00:00<?, ?it/s]

Density tuning:   0%|          | 0/2109 [00:00<?, ?it/s]

Density seed=42 epoch=4:   0%|          | 0/6816 [00:00<?, ?it/s]

...

Density tuning:   0%|          | 0/2109 [00:00<?, ?it/s]

Density seed=42 epoch=15:   0%|          | 0/6816 [00:00<?, ?it/s]

Density tuning:   0%|          | 0/2109 [00:00<?, ?it/s]

Held-out density quality:   0%|          | 0/1500 [00:00<?, ?it/s]

YOLO11n/rgb_control seed=42 epoch=1:   0%|          | 0/150948 [00:00<?, ?it/s]

## 5. Accuracy, speed and paired changes

COCO bbox mAP@[.50:.95], AP50, AR100, precision and recall at confidence .25/IoU .5 use one evaluator, seven shared categories and a maximum of 100 detections per image. Dataset scores remain separate. Per-class AP is blank when a class lacks ground truth.

nuScenes labels are 2D projections of 3D boxes using camera calibration and convex-hull clipping. They are an imperfect image-box proxy, not the official nuScenes 3D score. Foggy Cityscapes uses instance-mask box rectangles, merges riders into person and retains crowd annotations for scoring. Crowd-bearing training images are excluded for all matched arms.

Every model sees the same square resize (640 by default). This deliberately departs from native aspect-ratio preprocessing and must be disclosed. Matched comparisons isolate the supplied scalar under this training recipe; they do not guarantee an improvement or establish driving safety.


In [ ]:
if not results.empty:
    display(results[results.density == 'all'].round(4))
    paired_path = Path(CONFIG['output'])/'paired-deltas.csv'
    paired = pd.read_csv(paired_path, dtype={'density':str})
    display(paired.loc[paired.density == 'all', ['model','dataset','seed','arm','delta_mAP','delta_latency_ms']].round(4))
    density_quality = pd.read_csv(Path(CONFIG['output'])/'density-quality.csv')
    if not density_quality.empty: display(density_quality.round(6))
    else: print('No held-out known-beta labels: density error is unmeasured on native nuScenes.')
else:
    print('No full-study results are available in this notebook execution.')

## 6. Figures and interpretation

Plots show accuracy versus latency, per-class accuracy, known fog-level performance when available, paired deltas and training/tuning curves. Each point is a seed; summary standard deviation is between-seed spread, not a confidence interval. A single seed or tiny validation fixture cannot establish significance. Native nuScenes has unknown ground-truth density, so no known-density curve is invented for it.


In [ ]:
if not results.empty:
    figure_dir = Path(CONFIG['output'])/'figures'
    for pattern in ('*-comparison.png','*-deltas.png','*-classes.png','*-density.png','*-loss.png'):
        for path in sorted(figure_dir.glob(pattern)):
            display(Image(filename=str(path), width=1000))

## Reproducibility and sources

Reports contain the exact input manifest, configuration, package versions, input identity/stat fingerprint, source hashes, trained checkpoints, loss histories, raw predictions/timings, status, per-seed tables, paired deltas and PNG figures. Source images remain external and must be preserved; metadata fingerprints do not replace a byte-for-byte dataset checksum.

Run `python scripts/run_fog_research.py analyze --output <run-folder>` to regenerate analytics from completed runs. Completed model results remain inspectable if a later model fails. Model weights, optimizers and image data can require substantial disk space.

- [Foggy Cityscapes data and beta units](https://people.ee.ethz.ch/~csakarid/SFSU_synthetic/)
- [Ultralytics YOLO11](https://docs.ultralytics.com/models/yolo11/)
- [Official DETR weights and no_timm revision](https://huggingface.co/facebook/detr-resnet-50)
- [nuScenes devkit projection reference](https://github.com/nutonomy/nuscenes-devkit/blob/master/python-sdk/nuscenes/scripts/export_2d_annotations_as_json.py)

The earlier CMPE proposal is context; the current user request governs models, data and the full-training scope.
